## データ読込み

In [ ]:
import pickle
import numpy as np
import pandas as pd
from vinecopulas.marginals import *
from vinecopulas.vinecopula import *

# Per-unit execution-time sample lists
u1_sample = 'sample/u0101.pkl'
u2_sample = 'sample/u0102.pkl'
# Per-unit pre-estimated pWCET distributions {exceed_prob: WCET}
u1_pwcet = 'pwcet/u0101.pkl'
u2_pwcet = 'pwcet/u0102.pkl'

with open(u1_sample, 'rb') as f:
    u1_sample = pickle.load(f)
with open(u2_sample, 'rb') as f:
    u2_sample = pickle.load(f)
with open(u1_pwcet, 'rb') as f:
    u1_pwcet = pickle.load(f)
with open(u2_pwcet, 'rb') as f:
    u2_pwcet = pickle.load(f)

df = pd.DataFrame({'u1_sample': u1_sample, 'u2_sample': u2_sample})

## コピュラによるモデリング

### 擬似観測値生成

時系列サンプルの分布を[0, 1]領域に変換

In [ ]:
data_matrix = np.array(df)
u = pseudodata(data_matrix)

### コピュラをフィッティング

In [ ]:
copula_families = list(range(1, 16))
M, P, C = fit_vinecop(u, copula_families, vine='R')

** Tree:  1
0,1  --->  Student : parameters =  [-0.9786817784445127, 26.175681762582048]
Fitted copula families: [[ 1.  1.]
 [ 0. nan]]
Pair copula parameters: [[list([-0.9786817784445127, 26.175681762582048]) nan]
 [nan nan]]
Pair copula structures: [[15. nan]
 [nan nan]]


### コピュラからランダムサンプル生成

In [ ]:
n_sims = 1e7
simulated_uniforms = random(15, [-0.9786817784445127, 26.175681762582048], n_sims) # structure, parameters, n_sims

### [0, 1]領域から元のスケールに変換

In [ ]:
def build_icdf_from_pwcet_dict(pwcet_dict):
    alphas = np.array(sorted(pwcet_dict.keys()), dtype=float)
    xs_alpha = np.array([pwcet_dict[alpha] for alpha in alphas], dtype=float)

    ps = 1.0 - alphas[::-1]
    xs = xs_alpha[::-1]

    def icdf(u):
        u = np.asarray(u, dtype=float)
        return np.interp(u, ps, xs)
    return icdf

In [ ]:
u1_icdf = build_icdf_from_pwcet_dict(u1_pwcet)
u2_icdf = build_icdf_from_pwcet_dict(u2_pwcet)

uv = np.asarray(simulated_uniforms, dtype=float)
if uv.shape[0] == 2 and uv.shape[1] != 2:
    uv = uv.T

x1 = u1_icdf(uv[:, 0])
x2 = u2_icdf(uv[:, 1])
sums = x1 + x2

## 結果分布を出力

In [ ]:
def samples_to_pwcet_dict(samples, exceed_probs):
    samples = np.sort(np.asarray(samples, dtype=float))
    n = len(samples)
    out = {}

    for alpha in exceed_probs:
        idx = int(np.ceil((1 - alpha) * n)) - 1
        idx = min(max(idx, 0), n - 1)
        out[alpha] = samples[idx]
    return out

In [ ]:
target_alphas = sorted(set(u1_pwcet.keys()) & set(u2_pwcet.keys()), reverse=True)
sum_pwcet = samples_to_pwcet_dict(sums, target_alphas)